In [ ]:
import pandas as pd
import csv
# get working directory
import os
import simpledorff
from simpledorff import data_transforms, metrics

os.getcwd()
# set working directory
os.chdir('analyses/annotation/reliability')

In [ ]:
# Import csv file
df = pd.read_csv('reliability_actors_final_extra.csv', delimiter = ';',  quoting=csv.QUOTE_NONNUMERIC, encoding='utf-8')
print(df.shape)
# make article_id int
df['article_id'] = df['article_id'].astype(int)

In [ ]:
df.head()

In [ ]:
# drop if actors_present is 0 
df = df[df['actors_present'] == 1]
print(df.shape)

In [ ]:
# any duplicates based on article_id and coder? 
df.duplicated(subset=['article_id', 'coder', 'actor_name']).sum()

duplicated = df[df.duplicated(subset=['article_id', 'coder', 'actor_name'], keep=False)] 
duplicated

In [ ]:
print(df.shape)
df.head()

In [ ]:
# clean column actor_name, remove leading and trailing whitespaces and make it uppercase
df['actor_name'] = df['actor_name'].str.strip()
df['actor_name'] = df['actor_name'].str.upper()
df['actor_name'].value_counts()

In [ ]:
# sort df by article_id, coder, actor_name
df = df.sort_values(by=['article_id', 'actor_name', 'coder'])
df.head()

In [ ]:
# save df to csv
df.to_csv('reliability_actors_final_cleaned.csv', sep=';', index=False, quoting=csv.QUOTE_NONNUMERIC, encoding='utf-8')

## Create a df where we calculate the number of actors coded by each coder for each article

In [ ]:
# get article_id's coded by both coders
article_ids_coded = df.groupby('article_id')['coder'].nunique().reset_index()
article_ids_coded.coder.value_counts()

In [ ]:
article_ids_coded[article_ids_coded['coder'] == 1]

df[df['article_id'].isin(article_ids_coded[article_ids_coded['coder'] == 1]['article_id'])]

df = df[~df['article_id'].isin(article_ids_coded[article_ids_coded['coder'] == 1]['article_id'])]
print(df.shape)

In [ ]:
df.article_id.nunique()

In [ ]:
nr_actors_percoder = df.groupby(['coder', 'article_id'])['actor_name'].nunique().reset_index()
nr_actors_percoder = nr_actors_percoder.sort_values(by=['article_id', 'coder'])
nr_actors_percoder.head()

In [ ]:
# nr actors per coder per actor_type
nr_actors_peractor = df.groupby(['coder', 'actor_type', 'article_id'])['actor_name'].nunique().reset_index()
nr_actors_peractor = nr_actors_peractor.sort_values(by=['article_id', 'coder'])
nr_actors_peractor.head()

# make it a long df format
nr_actors_peractor = nr_actors_peractor.pivot_table(index=['coder', 'article_id'], columns='actor_type', values='actor_name').reset_index()
nr_actors_peractor = nr_actors_peractor.fillna(0)
nr_actors_peractor.columns = ['coder', 'article_id', 'nr_location', 'nr_organization', 'nr_person']
nr_actors_peractor

In [ ]:
nr_actors_percoder.actor_name.value_counts(dropna=False)

In [ ]:
print(simpledorff.calculate_krippendorffs_alpha_for_df(nr_actors_percoder,experiment_col=['article_id'],
                                                    annotator_col='coder',
                                                    class_col="actor_name",
                                                    metric_fn=metrics.interval_metric))

In [ ]:
print(simpledorff.calculate_krippendorffs_alpha_for_df(nr_actors_peractor,experiment_col=['article_id'],
                                                    annotator_col='coder',
                                                    class_col="nr_organization",
                                                    metric_fn=metrics.interval_metric))

In [ ]:
print(simpledorff.calculate_krippendorffs_alpha_for_df(nr_actors_peractor,experiment_col=['article_id'],
                                                    annotator_col='coder',
                                                    class_col="nr_person",
                                                    metric_fn=metrics.interval_metric))

## Get actors that are coded by both coders

In [ ]:
# get df for second_coder and main_coder

second_coder_df = df[df['coder'] == 'second_coder']
main_coder_df = df[df['coder'] == 'main_coder']

print(second_coder_df.shape, main_coder_df.shape)

In [ ]:
second_coder_df.columns

# drop title, about_covid, actors_present from both dfs
second_coder_df = second_coder_df.drop(columns=['title', 'about_covid', 'actors_present'])
main_coder_df = main_coder_df.drop(columns=['title', 'about_covid', 'actors_present'])

In [ ]:
# clean up actor_name column in both dfs
second_coder_df['actor_name'] = second_coder_df['actor_name'].str.strip()
second_coder_df['actor_name'] = second_coder_df['actor_name'].str.upper()

main_coder_df['actor_name'] = main_coder_df['actor_name'].str.strip()
main_coder_df['actor_name'] = main_coder_df['actor_name'].str.upper()

# remove special characters from actor_name column
second_coder_df['actor_name'] = second_coder_df['actor_name'].str.replace("(", "")
second_coder_df['actor_name'] = second_coder_df['actor_name'].str.replace(")", "")
main_coder_df['actor_name'] = main_coder_df['actor_name'].str.replace("(", "")
main_coder_df['actor_name'] = main_coder_df['actor_name'].str.replace(")", "")

# remove extra whitespaces from actor_name column
second_coder_df['actor_name'] = second_coder_df['actor_name'].str.replace("  ", " ")
main_coder_df['actor_name'] = main_coder_df['actor_name'].str.replace("  ", " ")

# sort both dfs by article_id, actor_name
second_coder_df = second_coder_df.sort_values(by=['article_id', 'actor_name'])
main_coder_df = main_coder_df.sort_values(by=['article_id', 'actor_name'])

In [ ]:
second_coder_df[second_coder_df['article_id'] == 2328119]

In [ ]:
from fuzzywuzzy import process
# Function to fuzzy match actor names within the same article_id
def fuzzy_merge(df1, df2, key1, key2, threshold):
    s = df2[key2].tolist()
    
    matches = df1[key1].apply(lambda x: process.extractOne(x, s, score_cutoff=threshold))
    df1['match'] = matches
    df1['match_name'] = df1['match'].apply(lambda x: x[0] if x else None)
    df1['match_score'] = df1['match'].apply(lambda x: x[1] if x else None)
    
    df_merged = df1.merge(df2, left_on=['article_id', 'match_name'], right_on=['article_id', 'actor_name'], how='outer')
    
    return df_merged

# fuzzy match second_coder_df and main_coder_df
df_merged = fuzzy_merge(second_coder_df, main_coder_df, 'actor_name', 'actor_name', 80)


In [ ]:
print(df_merged.shape)

In [ ]:
for i in df_merged.columns:
    print(i)

In [ ]:
df_merged.match_score.value_counts(dropna=False)

In [ ]:
second_coder_df[second_coder_df['article_id'] == 2328119]

In [ ]:
# see where match_score is NaN
df_merged[df_merged['match_score'].isna()][['actor_name_x', 'actor_name_y', 'article_id']]

In [ ]:
df_merged.isnull().sum()

In [ ]:
# write df_merged to excel
df_merged.to_excel('reliability_actors_final_cleaned_fuzzy.xlsx', index=False)

In [ ]:
# reread corrected df
df_merged = pd.read_excel('reliability_actors_final_cleaned_fuzzy_corrected.xlsx')
df_merged.head()

In [ ]:
# create a unique id for each row
df_merged['unique_actor_id'] = df_merged.groupby(['article_id']).cumcount() + 1

In [ ]:
df_merged['unique_actor_id'].value_counts(dropna=False)

# concat article_id and unique_actor_id to create unique_actor_id
df_merged['unique_actor_id'] = df_merged['article_id'].astype(str) + '_' + df_merged['unique_actor_id'].astype(str)

df_merged['unique_actor_id'].value_counts(dropna=False)

In [ ]:
# divide df's again, get article_id, unique_actor_id and every column that ends with _x
second_coder_df = df_merged[['article_id', 'unique_actor_id'] + [col for col in df_merged.columns if col.endswith('_x')]]
# remove _x from column names
second_coder_df.columns = second_coder_df.columns.str.replace('_x', '')

main_coder_df = df_merged[['article_id', 'unique_actor_id'] + [col for col in df_merged.columns if col.endswith('_y')]]
# remove _y from column names
main_coder_df.columns = main_coder_df.columns.str.replace('_y', '')

In [ ]:
len(second_coder_df.columns), len(main_coder_df.columns)

In [ ]:
print(second_coder_df.shape, main_coder_df.shape)

In [ ]:
second_coder_df.head()

# # drop article_id from both dfs
# second_coder_df = second_coder_df.drop(columns=['article_id'])
# main_coder_df = main_coder_df.drop(columns=['article_id'])

# concat both dfs
df_concatted = pd.concat([second_coder_df, main_coder_df], axis=0)

In [ ]:
print(df_concatted.shape)

df_concatted.head()

In [ ]:
df_concatted.coder.value_counts(dropna=False)

In [ ]:
df_concatted.unique_actor_id.value_counts(dropna=False)

# drop actor_name
df_concatted = df_concatted.drop(columns=['actor_name'])

In [ ]:
# copy df_concatted to actors_df_final
actors_df_final = df_concatted.copy()

In [ ]:
# write this to an excel file
actors_df_final.to_excel('reliability_actors_final_df.xlsx', index=False)

In [ ]:
actors_df_final.coder.value_counts(dropna=False)

## Calculate alpha for all actor columns

In [ ]:
for i in actors_df_final.columns:
    print(i)

In [ ]:
for i in actors_df_final.columns:
    print(i)

In [ ]:
print(simpledorff.calculate_krippendorffs_alpha_for_df(actors_df_final,
                                                    experiment_col='unique_actor_id',
                                                    annotator_col='coder',
                                                    class_col='actor_type'))

In [ ]:
print(simpledorff.calculate_krippendorffs_alpha_for_df(actors_df_final,
                                                    experiment_col='unique_actor_id',
                                                    annotator_col='coder',
                                                    class_col='actor_function'))

In [ ]:
print(simpledorff.calculate_krippendorffs_alpha_for_df(actors_df_final,
                                                    experiment_col='unique_actor_id',
                                                    annotator_col='coder',
                                                    class_col='actor_pp'))

In [ ]:
print(simpledorff.calculate_krippendorffs_alpha_for_df(actors_df_final,
                                                    experiment_col='unique_actor_id',
                                                    annotator_col='coder',
                                                    class_col='directly_quoted'))

In [ ]:
print(simpledorff.calculate_krippendorffs_alpha_for_df(actors_df_final,
                                                    experiment_col='unique_actor_id',
                                                    annotator_col='coder',
                                                    class_col='indirectly_quoted'))

In [ ]:
print(simpledorff.calculate_krippendorffs_alpha_for_df(actors_df_final,
                                                    experiment_col='unique_actor_id',
                                                    annotator_col='coder',
                                                    class_col='nr_words',
                                                    metric_fn=metrics.interval_metric))

In [ ]:
print(simpledorff.calculate_krippendorffs_alpha_for_df(actors_df_final,
                                                    experiment_col='unique_actor_id',
                                                    annotator_col='coder',
                                                    class_col='talks_covid_measures'))

In [ ]:
actors_df_final.head()

In [ ]:
actors_df_final.talks_covid_measures.value_counts(dropna=False)

In [ ]:
# get unique_actor_id's that are coded 1 in talks_covid_measures
actors_measures = actors_df_final[actors_df_final['talks_covid_measures'] == 1]['unique_actor_id'].unique()
print(len(actors_measures))

# get df where talks_covid_measures is 1
covid_measures_df = actors_df_final[actors_df_final['talks_covid_measures'] == 1]

# count the number of actors coded by both coders as 1 in talks_covid_measures
covid_measures_df = covid_measures_df[covid_measures_df['unique_actor_id'].isin(actors_measures)]

# count the number of actors coded per coder
final_ids_measures = covid_measures_df.groupby('unique_actor_id')['coder'].count().reset_index(name='count').sort_values(by='count', ascending=False)

In [ ]:
# final_ids to keep
final_ids = final_ids_measures[final_ids_measures['count'] == 2]['unique_actor_id'].unique()

# get df where unique_actor_id is in final_ids
covid_measures_df = covid_measures_df[covid_measures_df['unique_actor_id'].isin(final_ids)]
covid_measures_df.shape

In [ ]:
covid_measures_df.head()

In [ ]:
covid_measures_df.coder.value_counts(dropna=False)

In [ ]:
cov_measures_vars = ['measure_1', 'measure_2', 'measure_3', 'measure_4', 'measure_5',
                        'measure_6', 'measure_7', 'measure_8', 'measure_9', 'measure_10',
                        'measure_11', 'measure_12', 'measure_13', 'measure_14', 'measure_15',
                        'measure_16', 'measure_17', 'measure_other']

# create a variable all measures, if actor talks about any measure set it to 1 else 0
covid_measures_df['all_measures'] = covid_measures_df[cov_measures_vars].max(axis=1)
covid_measures_df['all_measures'].value_counts(dropna=False)

In [ ]:
for i in df.columns:
    if i in cov_measures_vars:
        print(covid_measures_df[i].value_counts(dropna=False))

In [ ]:
for i in df.columns:
    if i in cov_measures_vars:
        print(covid_measures_df[covid_measures_df['coder'] == 'second_coder'][i].value_counts(dropna=False))

In [ ]:
for i in df.columns:
    if i in cov_measures_vars:
        print(covid_measures_df[covid_measures_df['coder'] == 'main_coder'][i].value_counts(dropna=False))

In [ ]:
for i in covid_measures_df.columns:
    if i in cov_measures_vars:
        print(covid_measures_df[i].value_counts(dropna=False))
        try:
            print('krippendorfs alpha is:', simpledorff.calculate_krippendorffs_alpha_for_df(covid_measures_df,experiment_col='unique_actor_id',
                                                        annotator_col='coder',
                                                        class_col=i))
        except:
            print('error')
            pass

In [ ]:
cov_measures_vars_included = ['measure_1', 'measure_15', 'measure_17']

cov_measures_pos_vars = ['measure_1_positive', 'measure_15_positive', 'measure_17_positive']

cov_measures_neg_vars = ['measure_1_negative', 'measure_15_negative', 'measure_17_negative']
